# 4장 — 차원의 저주 🌀 (핵심)

고차원의 이상함을 **숫자로** 확인하고, 그 이상함이 검색을 가능하게 하는 반전까지 봅니다.

**이 노트북의 목표**
1. 무작위 벡터 쌍의 코사인이 차원과 함께 0으로 집중(σ ≈ 1/√d)됨을 확인
2. 거리 집중: 최근접과 최원접의 차이가 줄어드는 것을 확인
3. 반전: 학습된(관련지어진) 쌍은 무작위 바닥 위로 뚜렷이 솟는다

> 시드 고정(결정적) + 차원·표본 수를 작게 잡아 즉시 돕니다.

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch4')

## 1. 실험 ①: 무작위 쌍의 코사인, 차원별로

$$\text{무작위 쌍의 } \cos\theta \approx 0 \pm \tfrac{1}{\sqrt d}$$

In [ ]:
import numpy as np
rng = np.random.default_rng(42)   # 시드 고정 — 항상 같은 결과

def rand_unit(d, n=1):
    v = rng.standard_normal((n, d))
    return v / np.linalg.norm(v, axis=1, keepdims=True)

print(' 차원 d   σ(실측)   1/√d(이론)   |cos| 최대 (300쌍)')
stds = []
for d in [2, 8, 32, 128, 512]:
    cos = np.sum(rand_unit(d, 300) * rand_unit(d, 300), axis=1)
    stds.append(cos.std())
    print(f'   {d:4d}   {cos.std():.3f}     {1/np.sqrt(d):.3f}        {np.abs(cos).max():.2f}')
    assert cos.std() < 2 / np.sqrt(d), 'σ는 1/√d 수준이어야 (집중 현상)'

assert stds == sorted(stds, reverse=True), '차원이 오를수록 σ는 줄어야'
print('\n✔ 확인: 차원이 오를수록 무작위 코사인이 0(직교)으로 조여든다 (3부 7장 큰 수의 법칙)')

## 2. 실험 ② · 거리 집중: 모두가 엇비슷하게 멀다

"가장 가까운 점"과 "가장 먼 점"의 차이가 고차원에서 얼마나 미미해지는지 봅니다.

In [ ]:
print(' 차원 d   (최원접 − 최근접) / 최근접')
gaps = []
for d in [2, 32, 512]:
    pts = rng.standard_normal((200, d))          # 무작위 점 200개
    q = rng.standard_normal(d)                   # 기준점
    dist = np.linalg.norm(pts - q, axis=1)
    gap = (dist.max() - dist.min()) / dist.min()
    gaps.append(gap)
    print(f'   {d:4d}   {gap:.2f}  (최근접 {dist.min():.1f} ~ 최원접 {dist.max():.1f})')

assert gaps[0] > gaps[-1] * 2, '고차원일수록 최근접·최원접의 상대 차이가 줄어야 — 거리 집중'
print('\n⚠️ 512차원: 무작위 점들 사이에선 "가장 가까움"이 흐릿하다. 이것이 차원의 저주')

## 3. 실험 ③ · 반전: 신호는 바닥 위로 솟는다

저주의 전제는 **무작위**였습니다. 임베딩은 무작위가 아니라 **학습된** 벡터입니다.
관련 문서를 흉내 내(질문 + 잡음) 무작위 무더기와 비교합니다.

In [ ]:
d = 256
q = rand_unit(d)[0]                              # 질문 벡터
related = q + 1.1 * rand_unit(d)[0]              # 학습으로 관련지어진 문서 (질문 + 잡음)
related /= np.linalg.norm(related)

signal = q @ related
floor = np.abs(rand_unit(d, 5000) @ q)           # 무작위 문서 5,000개의 점수

print(f'관련 문서의 점수      : {signal:.2f}')
print(f'무작위 5,000개 중 최고: {floor.max():.2f}  (바닥 폭 1/√{d} ≈ {1/np.sqrt(d):.3f})')
print(f'신호는 바닥 폭의 약 {signal / (1/np.sqrt(d)):.0f}σ, 우연으로는 불가능한 값')

assert signal > floor.max() + 0.2, '학습된 신호는 무작위 최고점보다 확실히 높아야'
assert floor.max() < 0.35, '무작위는 수천 개를 던져도 바닥(0 근처)을 못 벗어난다'
print('\n✔ 확인: 무작위는 저주 덕에 0에 깔리고, 학습된 관계만 솟는다 (임베딩 검색이 작동하는 이유)')

## 4. 관찰 정리 + 🤖 AI 연결

- **집중**: 무작위 쌍의 cos ≈ 0 ± 1/√d. 성분 곱 d개의 합이라 큰 수의 법칙(3부 7장)이 작동.
- **거리 집중**: 무작위 점들끼리는 최근접·최원접 차이가 미미. 절대 점수보다 **순위**가 믿을 만(7장).
- **반전**: 임베딩은 무작위가 아니다. 무관한 쌍은 0 바닥에 깔리고(저주 = 잡음 제거기),
  학습된 관계만 그 위로 솟는다. cos 0.3도 바닥 대비 수 σ짜리 강한 신호일 수 있다.
- **용량**: 거의 직교 방향이 지수적으로 많다(1부 9장 차원의 확장). 수백만 개념을 담는 도서관.

**다음 챕터**: 5장, 지도 그리기(차원 축소). 눈에 안 보이는 수백 차원 지도를 종이에 펴는 법:
1부 8장의 **PCA**와 재회하고, t-SNE·UMAP 지도의 **왜곡을 읽는 법**을 배웁니다.